In [ ]:
class GaloisField:

    def __init__(self, m_order, primitive_poly):
        """
        Elements and operations to work in Galois fields

        Args:
            m_order: Field order M
            primitive_poly: Bitfield representation of primitive poly powers. Ex: X**4 + X + 1 == 0b10011
        """
        self._m_order           = m_order
        self._primitive_poly    = primitive_poly
        self._elements          = []
        self.generate_field()
        self._zero              = self._elements.index(0)
        self._cycle             = 2**self._m_order - 1

    def __call__(self, n):
        try:
            return self._elements[n]
        except:
            return self._elements[n % self._cycle]

    def __str__(self):
        info = f"{'='*128}\n"
        for e in self._elements: info += f"{e:0{self._m_order}b}--{self._elements.index(e)}\n"
        info += f"{'='*128}\n"

        return info

    def get_poly(self,e):
        """
        Get polynomial representation of a element

        Args:
            e: Field element
        """
        return [(e>>b) & 1 for b in range(self._m_order+1)]

    def generate_field(self):
        """
        Generate all the field elements for the given order and primitive poly. Zero element always added last
        """

        for i in range(2**self._m_order):
            # get all primitive elements (1, alpha, alpha**2..)
            if i < self._m_order:
                alpha_n_final = 1<<i
            # get the m element (alpha**m)
            elif i == self._m_order:
                alpha_n_final = self._primitive_poly & (2**self._m_order)-1
            # get the remaining elements
            else:
                # multiply last one by alpha (LRL), get polynomial representation, simplify
                alpha_n         = self._elements[-1]<<1
                alpha_n_poly    = self.get_poly(alpha_n)

                # simplify using know elements
                alpha_n_final = 0
                for i in range(len(alpha_n_poly)):
                    if alpha_n_poly[i]:
                        alpha_n_final ^= self._elements[i]

            if i > 0 and alpha_n_final==1:
                print(f"Field Closed Succesfully!, {len(self._elements)} Non-Zero Elements")
                break
            else:
                self._elements += [alpha_n_final]

        # complete the field with zero element
        self._elements += [0]
    
    def add(self, a, b):
        ans = self(a) ^ self(b)
        i   = self._elements.index(ans)
        return ans, i

    def mul(self, a, b):
        if a == self._zero or b == self._zero:
            i   = self._zero
        else:
            i   = (a+b) % self._cycle

        ans = self(i)
        return ans, i

    def inv(self, a):
        if a == self._zero:
            i   = a
            trow: ZeroDivisionError
        else:
            i   = (self._cycle - a) % self._cycle

        ans = self(i)
        return ans, i

    def div(self, a, b):
            if b == self._zero:
                i   = b
                trow: ZeroDivisionError
            else:
                ans, i = self.mul(a, self.inv(b)[0])
    
            return ans, i
        
    


In [67]:
m_order = 4
primitive_poly = 0b11001

gf = GaloisField(m_order, primitive_poly)

ZERO = gf._zero

print(gf)

gf(3)
gf(25)

gf.add(ZERO,3)
gf.mul(ZERO,3)
gf.mul(7,3)
gf.mul(14,3)
gf.inv(1)
gf.inv(ZERO)
gf.inv(3)
gf.div(0,3)



Field Closed Succesfully!, 15 Non-Zero Elements
0001--0
0010--1
0100--2
1000--3
1001--4
1011--5
1111--6
0111--7
1110--8
0101--9
1010--10
1101--11
0011--12
0110--13
1100--14
0000--15



(8, 3)